# Dendritic f-prime Perturbation Check

This notebook checks the small-perturbation identity behind the simplified dendritic error idea:

$$f'(x) \approx \frac{f(x+\delta x)-f(x-\delta x)}{2\delta x}.$$

For a hidden unit with basal potential $a$ and apical feedback direction $b$, the central difference

$$\frac{f(a+\epsilon b)-f(a-\epsilon b)}{2\epsilon}$$

should approximate $f'(a)b$. If $b = W_{out}^T(p-y)$, this matches the standard BP hidden error term for a one-hidden-layer network.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## Tiny Network

Use a single-hidden-layer classifier:

$$x \rightarrow a = W_{in}x \rightarrow h=f(a) \rightarrow z=W_{out}h \rightarrow \mathrm{softmax}(z).$$

For cross entropy, the output error is $e = p-y$. Backprop gives hidden pre-activation error:

$$\delta_h^{BP} = f'(a) \odot W_{out}^T e.$$

The dendritic finite-difference estimate uses the apical feedback $b=W_{out}^T e$:

$$\delta_h^{FD}(\epsilon)=\frac{f(a+\epsilon b)-f(a-\epsilon b)}{2\epsilon}.$$

In [ ]:
def softmax(z):
    z = z - np.max(z)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z)

def f(x):
    return np.tanh(x)

def fprime_analytic(x):
    return 1.0 - np.tanh(x) ** 2

def cosine(a, b):
    return float(np.dot(a.ravel(), b.ravel()) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

def rel_error(a, b):
    return float(np.linalg.norm(a - b) / (np.linalg.norm(b) + 1e-12))

input_dim = 7
hidden_dim = 11
output_dim = 5

x = rng.normal(size=input_dim)
target = 2
y = np.zeros(output_dim)
y[target] = 1.0

W_in = rng.normal(scale=0.4, size=(hidden_dim, input_dim))
W_out = rng.normal(scale=0.4, size=(output_dim, hidden_dim))

a = W_in @ x
h = f(a)
logits = W_out @ h
p = softmax(logits)
output_error = p - y

apical_feedback = W_out.T @ output_error
bp_hidden_error = fprime_analytic(a) * apical_feedback
bp_W_in_grad = np.outer(bp_hidden_error, x)

print('target:', target)
print('probs:', p)
print('CE loss:', -np.log(p[target] + 1e-12))

## Central-Difference Check

Sweep the perturbation size $\epsilon$. Large $\epsilon$ has Taylor approximation error; extremely small $\epsilon$ can suffer floating-point cancellation. A middle range should closely match BP.

In [ ]:
eps_values = np.logspace(-1, -8, 8)
rows = []

for eps in eps_values:
    fd_hidden_error = (f(a + eps * apical_feedback) - f(a - eps * apical_feedback)) / (2.0 * eps)
    fd_W_in_grad = np.outer(fd_hidden_error, x)
    rows.append({
        'eps': eps,
        'hidden_rel_error': rel_error(fd_hidden_error, bp_hidden_error),
        'hidden_cosine': cosine(fd_hidden_error, bp_hidden_error),
        'grad_rel_error': rel_error(fd_W_in_grad, bp_W_in_grad),
        'grad_cosine': cosine(fd_W_in_grad, bp_W_in_grad),
    })

for row in rows:
    print(
        f"eps={row['eps']:.0e} "
        f"hidden_rel={row['hidden_rel_error']:.3e} "
        f"hidden_cos={row['hidden_cosine']:.6f} "
        f"grad_rel={row['grad_rel_error']:.3e} "
        f"grad_cos={row['grad_cosine']:.6f}"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].loglog([r['eps'] for r in rows], [r['hidden_rel_error'] for r in rows], marker='o', label='hidden error')
axes[0].loglog([r['eps'] for r in rows], [r['grad_rel_error'] for r in rows], marker='s', label='W_in grad')
axes[0].invert_xaxis()
axes[0].set_xlabel('epsilon')
axes[0].set_ylabel('relative error vs BP')
axes[0].legend(frameon=False)

axes[1].semilogx([r['eps'] for r in rows], [r['hidden_cosine'] for r in rows], marker='o', label='hidden error')
axes[1].semilogx([r['eps'] for r in rows], [r['grad_cosine'] for r in rows], marker='s', label='W_in grad')
axes[1].invert_xaxis()
axes[1].set_xlabel('epsilon')
axes[1].set_ylabel('cosine similarity vs BP')
axes[1].set_ylim(0.99, 1.001)
axes[1].legend(frameon=False)

fig.tight_layout()
plt.show()

## Raw Difference vs Derivative

Without dividing by $2\epsilon$, the raw dendritic firing-rate contrast is

$$f(a+\epsilon b)-f(a-\epsilon b) \approx 2\epsilon f'(a)b.$$

So the raw contrast has the same direction as BP, but its magnitude is scaled by $2\epsilon$.

In [ ]:
eps = 1e-3
raw_contrast = f(a + eps * apical_feedback) - f(a - eps * apical_feedback)
scaled_bp = 2.0 * eps * bp_hidden_error

print('raw contrast rel error vs 2 eps BP:', rel_error(raw_contrast, scaled_bp))
print('raw contrast cosine vs BP:', cosine(raw_contrast, bp_hidden_error))

## Random Feedback Control

The match above is strong because the apical feedback direction uses $W_{out}^T e$. If feedback is random, the same finite-difference mechanism still estimates $f'(a)b$, but $b$ is no longer the BP error direction.

In [ ]:
B_random = rng.normal(scale=0.4, size=(hidden_dim, output_dim))
random_feedback = B_random @ output_error
fd_random_error = (f(a + eps * random_feedback) - f(a - eps * random_feedback)) / (2.0 * eps)

print('finite diff estimates fprime * random feedback')
print('rel error vs analytic fprime*random_feedback:', rel_error(fd_random_error, fprime_analytic(a) * random_feedback))
print('cosine vs BP hidden error:', cosine(fd_random_error, bp_hidden_error))

## Takeaway

The center-difference dendritic perturbation recovers the local factor $f'(a)b$ very accurately for small enough $\epsilon$. It becomes BP-equivalent only when the apical feedback direction $b$ carries the correct output-error projection, such as $W_{out}^T(p-y)$ or a well-aligned feedback approximation.